In [94]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [95]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.data.load_clean_data import load_all_clean_data

dfs = load_all_clean_data()

from src.data.get_dfs import get_dfs

(circuits,
constructors,
constructor_standings,
drivers,
driver_standings,
qualifying,
races,
results) = get_dfs(dfs)

In [96]:
ml_df = results.copy()

drivers = drivers.rename(columns={
    "name": "driver_name",
    "nationality": "driver_nationality"
})

constructors = constructors.rename(columns={
    "name": "constructor_name",
    "nationality": "constructor_nationality"
})

races = races.rename(columns={
    "date": "race_date",
    "time": "race_time"
})

circuits = circuits.rename(columns={
    "locality": "circuit_locality",
    "country": "circuit_country"
})

qualifying = qualifying.rename(columns={
    "position": "qualifying_position"
})

In [97]:
ml_df.shape

(25939, 13)

In [98]:
missing_dob = drivers[
    drivers["dob"].isnull()
]
# Inspect races for each driver
for x in missing_dob["driver_id"]:

    print(f"\nDriver ID: {x}")

    display(
        ml_df[
            ml_df["driver_id"] == x
        ]
    )


Driver ID: paul_aron


,season,round,driver_id,driver_name,constructor_id,grid,position,points,laps,status,time,fastest_lap,fastest_lap_rank



Driver ID: dino_beganovic


,season,round,driver_id,driver_name,constructor_id,grid,position,points,laps,status,time,fastest_lap,fastest_lap_rank



Driver ID: luke_browning


...

,season,round,driver_id,driver_name,constructor_id,grid,position,points,laps,status,time,fastest_lap,fastest_lap_rank



Driver ID: cian_shields


,season,round,driver_id,driver_name,constructor_id,grid,position,points,laps,status,time,fastest_lap,fastest_lap_rank



Driver ID: frederik_vesti


,season,round,driver_id,driver_name,constructor_id,grid,position,points,laps,status,time,fastest_lap,fastest_lap_rank


In [99]:
ml_df = ml_df.merge(
    drivers,
    on="driver_id",
    how="left"
)

In [100]:
ml_df = ml_df.merge(
    constructors,
    on="constructor_id",
    how="left"
)

display(ml_df.head())
print(ml_df.shape)

,season,round,driver_id,driver_name,constructor_id,grid,position,points,laps,status,time,fastest_lap,fastest_lap_rank,given_name,family_name,dob,driver_nationality,constructor_name,constructor_nationality
0,1950,1,farina,Nino Farina,alfa,1,1,9.0,70,Finished,2:13:23.600,NaN,NaN,Nino,Farina,1906-10-30,Italian,Alfa Romeo,Swiss
1,1950,1,fagioli,Luigi Fagioli,alfa,2,2,6.0,70,Finished,+2.600,NaN,NaN,Luigi,Fagioli,1898-06-09,Italian,Alfa Romeo,Swiss
2,1950,1,reg_parnell,Reg Parnell,alfa,4,3,4.0,70,Finished,+52.000,NaN,NaN,Reg,Parnell,1911-07-02,British,Alfa Romeo,Swiss
3,1950,1,cabantous,Yves Cabantous,lago,6,4,3.0,68,+2 Laps,NaN,NaN,NaN,Yves,Cabantous,1904-10-08,French,Talbot-Lago,French
4,1950,1,rosier,Louis Rosier,lago,9,5,2.0,68,+2 Laps,NaN,NaN,NaN,Louis,Rosier,1905-11-05,French,Talbot-Lago,French


(25939, 19)


In [101]:
ml_df = ml_df.merge(
    races,
    on=["season", "round"],
    how="left"
)

In [102]:
ml_df.shape


(25939, 22)

In [103]:
ml_df = ml_df.merge(
    circuits,
    on="circuit_id",
    how="left"
)

In [104]:
ml_df.shape


(25939, 25)

In [105]:
ml_df = ml_df.merge(
    qualifying,
    on=["season", "round", "driver_id", "constructor_id"],
    how="left"
)

In [106]:
from src.utils.timetoseconds import time_to_seconds
# Create missing indicators BEFORE filling

ml_df["missing_q1"] = (
    ml_df["q1"].isna()
).astype(int)

ml_df["missing_q2"] = (
    ml_df["q2"].isna()
).astype(int)

ml_df["missing_q3"] = (
    ml_df["q3"].isna()
).astype(int)


ml_df["q1_seconds"] = (
    ml_df["q1"]
    .apply(time_to_seconds)
)

ml_df["q2_seconds"] = (
    ml_df["q2"]
    .apply(time_to_seconds)
)

ml_df["q3_seconds"] = (
    ml_df["q3"]
    .apply(time_to_seconds)
)

# Fill missing qualifying session times
# using the slowest recorded time in each session

ml_df["q1_seconds"] = ml_df["q1_seconds"].fillna(
    ml_df["q1_seconds"].max()
)

ml_df["q2_seconds"] = ml_df["q2_seconds"].fillna(
    ml_df["q2_seconds"].max()
)

ml_df["q3_seconds"] = ml_df["q3_seconds"].fillna(
    ml_df["q3_seconds"].max()
)


In [107]:
ml_df.shape

(25939, 35)

In [108]:
ml_df.head()

,season,round,driver_id,driver_name,constructor_id,grid,position,points,laps,status,...,qualifying_position,q1,q2,q3,missing_q1,missing_q2,missing_q3,q1_seconds,q2_seconds,q3_seconds
0,1950,1,farina,Nino Farina,alfa,1,1,9.0,70,Finished,...,NaN,NaN,NaN,NaN,1,1,1,1002.64,132.47,129.776
1,1950,1,fagioli,Luigi Fagioli,alfa,2,2,6.0,70,Finished,...,NaN,NaN,NaN,NaN,1,1,1,1002.64,132.47,129.776
2,1950,1,reg_parnell,Reg Parnell,alfa,4,3,4.0,70,Finished,...,NaN,NaN,NaN,NaN,1,1,1,1002.64,132.47,129.776
3,1950,1,cabantous,Yves Cabantous,lago,6,4,3.0,68,+2 Laps,...,NaN,NaN,NaN,NaN,1,1,1,1002.64,132.47,129.776
4,1950,1,rosier,Louis Rosier,lago,9,5,2.0,68,+2 Laps,...,NaN,NaN,NaN,NaN,1,1,1,1002.64,132.47,129.776


In [109]:
missing_dob = ml_df[ml_df["driver_nationality"].isnull()]

# Display relevant columns
missing_dob[[
    "driver_id",
    "driver_nationality"
]]

,driver_id,driver_nationality


In [110]:
ml_df["dob"] = pd.to_datetime(ml_df["dob"])
ml_df["race_date"] = pd.to_datetime(ml_df["race_date"])

In [111]:
ml_df["driver_age"] = (
    ml_df["race_date"] - ml_df["dob"]
).dt.days / 365.25

In [112]:
ml_df["podium"] = (
    ml_df["position"] <= 3
).astype(int)

In [113]:
ml_df["win"] = (
    ml_df["position"] == 1
).astype(int)

In [114]:
ml_df["pole_position"] = (
    ml_df["grid"] == 1
).astype(int)

In [115]:
ml_df["front_row"] = (
    ml_df["grid"] <= 2
).astype(int)

In [116]:
# =========================================================
# Weighted points at circuit
# =========================================================

ml_df["weighted_points_at_circuit"] = (
    ml_df.groupby(["driver_id", "circuit_id"])["points"]
    .transform(
        lambda x: (
            x.shift(1)
             .fillna(0)
             .ewm(alpha=0.5, adjust=False)
             .mean()
        )
    )
)

# =========================================================
# Driver circuit delta
# =========================================================

driver_circuit_avg = (
    ml_df.groupby(["driver_id", "circuit_id"])["points"]
    .transform(
        lambda x: (
            x.shift(1)
             .expanding()
             .mean()
             .fillna(0)
        )
    )
)

driver_overall_avg = (
    ml_df.groupby("driver_id")["points"]
    .transform(
        lambda x: (
            x.shift(1)
             .expanding()
             .mean()
             .fillna(0)
        )
    )
)

ml_df["driver_circuit_delta"] = (
    driver_circuit_avg - driver_overall_avg
).fillna(0)

In [117]:
# Points scored in the previous 5 races

points_last_5 = (
    ml_df.groupby("driver_id")["points"]
    .transform(
        lambda x: (
            x.shift(1)
            .rolling(5, min_periods=1)
            .sum()
        )
    )
    .fillna(0)

)

# Add feature to dataframe

ml_df["points_last_5_races"] = points_last_5

In [118]:
# Current season driver points before current race

ml_df["current_season_points"] = (
    ml_df.groupby(["season", "driver_id"])["points"]
    .cumsum()
    - ml_df["points"]
)

In [119]:
# Total driver points per season

season_driver_points = (
    ml_df.groupby(["season", "driver_id"])["points"]
    .sum()
    .reset_index()
    .sort_values(["driver_id", "season"])
)

# Average points across previous seasons

season_driver_points["avg_points_per_season"] = (
    season_driver_points
    .groupby("driver_id")["points"]
    .transform(
        lambda x: (
            x.shift(1)
            .expanding()
            .mean()
        )
    )
    .fillna(0)
)

# Merge back

ml_df = ml_df.merge(
    season_driver_points[
        ["season", "driver_id", "avg_points_per_season"]
    ],
    on=["season", "driver_id"],
    how="left"
)

In [120]:
# Constructor points scored per race

constructor_round_points = (
    ml_df.groupby(
        ["season", "round", "constructor_id"]
    )["points"]
    .sum()
    .reset_index()
)

# Cumulative constructor points BEFORE current race

constructor_round_points[
    "constructor_points_current_season"
] = (
    constructor_round_points
    .groupby(["season", "constructor_id"])["points"]
    .cumsum()
    - constructor_round_points["points"]
)

# Merge back to main dataframe

ml_df = ml_df.merge(
    constructor_round_points[
        [
            "season",
            "round",
            "constructor_id",
            "constructor_points_current_season"
        ]
    ],
    on=["season", "round", "constructor_id"],
    how="left"
)

In [121]:
ml_df.shape

(25939, 46)

In [122]:
ml_df["grid"] = pd.to_numeric(
    ml_df["grid"],
    errors="coerce"
).astype("Int64")

ml_df["qualifying_position"] = pd.to_numeric(
    ml_df["qualifying_position"],
    errors="coerce"
).astype("Int64")

same_positions = (
    ml_df["qualifying_position"] == ml_df["grid"]
).sum()

matching_ratio = same_positions / len(ml_df)

print(f"Matching ratio: {matching_ratio:.2%}")

Matching ratio: 32.56%


In [123]:
# Rows where grid and qualifying_position do NOT match
mismatches = ml_df[
    ml_df["qualifying_position"].isna()
]

# View mismatching rows
print(mismatches[["driver_id","season","round", "grid", "qualifying_position"]])

print(ml_df["grid"].isna().sum())
print(ml_df["qualifying_position"].isna().sum())

# Optional: number of mismatches
print(f"\nTotal mismatches: {len(mismatches)}")





             driver_id  season  round  grid  qualifying_position
0               farina    1950      1     1                 <NA>
1              fagioli    1950      1     2                 <NA>
2          reg_parnell    1950      1     4                 <NA>
3            cabantous    1950      1     6                 <NA>
4               rosier    1950      1     9                 <NA>
...                ...     ...    ...   ...                  ...
23692  mick_schumacher    2021      5    20                 <NA>
25812        bortoleto    2025     21    18                 <NA>
25878   max_verstappen    2026      1    20                 <NA>
25887            sainz    2026      1    21                 <NA>
25889           stroll    2026      1    22                 <NA>

[14910 rows x 5 columns]
0
14910

Total mismatches: 14910


In [124]:
ml_df["qualifying_position"] = (
    ml_df["qualifying_position"]
    .fillna(ml_df["grid"])
)


In [125]:
categorical_columns = [
    "driver_id",
    "driver_name",
    "constructor_id",
    "status",
    "given_name",
    "family_name",
    "driver_nationality",
    "constructor_name",
    "constructor_nationality",
    "race_name",
    "circuit_id",
    "circuit_name",
    "circuit_locality",
    "circuit_country"
]

for col in categorical_columns:
    ml_df[col] = ml_df[col].astype("category")


ml_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25939 entries, 0 to 25938
Data columns (total 46 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   season                             25939 non-null  int64         
 1   round                              25939 non-null  int64         
 2   driver_id                          25939 non-null  category      
 3   driver_name                        25939 non-null  category      
 4   constructor_id                     25939 non-null  category      
 5   grid                               25939 non-null  Int64         
 6   position                           25939 non-null  int64         
 7   points                             25939 non-null  float64       
 8   laps                               25939 non-null  int64         
 9   status                             25939 non-null  category      
 10  time                               8354 non-n

In [126]:
missing_values = ml_df.isna().sum()

print(missing_values[missing_values > 0])

time                17585
fastest_lap         17154
fastest_lap_rank    17154
q1                  15072
q2                  19692
q3                  22044
dtype: int64


In [127]:
ml_df.to_csv(
    "../data/processed/master_ml_dataset.csv",
    index=False
)
